In [17]:
import numpy as np
import networkx as nx
from qiskit.quantum_info import SparsePauliOp
from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper
from collections import Counter
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate, CXGate

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import HighLevelSynthesis, InverseCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping


from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti, DecomposePauliZEvolution

In [3]:
n = 3
T = 4
extended_swap_strat = ExtendedSwapStrategy.from_grid(3, 4)
num_physical_qubits = extended_swap_strat._num_vertices
num_physical_qubits

12

In [ ]:
num_qubits = n*T
rng = np.random.default_rng(seed=1)
doubles = rng.choice(num_qubits, (20, 2))
quads = rng.choice(num_qubits, (2, 4), replace=False)
sixes = rng.choice(num_qubits, (1, 6), replace=False)

coeffs = rng.random(23)

def choice_to_pauli(choice: list[int], size: int) -> str:
    pauli = ['I'] * size
    for x in choice:
        pauli[size - x - 1] = 'Z'
    return ''.join(pauli)

hamiltonian = SparsePauliOp(
    [choice_to_pauli(c, num_qubits) for c in doubles] + [choice_to_pauli(c, num_qubits) for c in quads] + [choice_to_pauli(c, num_qubits) for c in sixes], 
    coeffs=coeffs
)
hamiltonian = hamiltonian.simplify()
hamiltonian = hamiltonian.sort(weight=True)

In [11]:
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph

filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N3_W4.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [2,1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, fraction_terms=0.5)

In [20]:
import pickle
with open('/lustre/scratch127/qpg/jc59/hubo/simulation.grid.compilation.test_N3_W4.extra0.four0.0.six1.0.pkl', 'rb') as f:
    data = pickle.load(f)

In [22]:
data.keys()

dict_keys(['full_hamiltonian', 'compiled_hamiltonian', 0, 1, 2, 3, 4, 5, 6, 7, 8, 10])

In [23]:
hamiltonian = data['compiled_hamiltonian']

In [8]:
from qiskit_qaoa.utils.hamiltonian_utils import hamiltonian_to_interactions


In [24]:
all_pauli_z = np.array(
    [i.paulis[0].z for i in hamiltonian]
)
print(f'Hamiltonian: {len(hamiltonian)}')
print(f'Orders: {Counter(np.sum(all_pauli_z, axis=1))}')

program_interactions = hamiltonian_to_interactions(hamiltonian, 0.0, 1.0)
lengths = Counter([len(interaction) for interaction in program_interactions])

print(f'Program interactions: {len(program_interactions)}')
print(f'Orders: {Counter([len(interaction) for interaction in program_interactions])}')

Hamiltonian: 163
Orders: Counter({np.int64(3): 56, np.int64(2): 46, np.int64(4): 34, np.int64(1): 12, np.int64(5): 12, np.int64(6): 2, np.int64(0): 1})
Program interactions: 136
Orders: Counter({3: 56, 2: 46, 4: 34})


In [42]:
program_interactions = [tuple(x) for x in program_interactions]
program_interactions.index((7,8,9))

9

In [43]:
program_interactions[9]

(np.int64(7), np.int64(8), np.int64(9))

In [25]:
mapper = HigherOrderSatMapper(timeout=30)
results = {}
num_layers = 8
sat_results = mapper.hubo_max_sat(
    num_qubits, program_interactions, extended_swap_strat, num_layers
)
mapping = sat_results[num_layers][1]
edge_map = dict(mapping)
print(f'Cost: {sat_results[num_layers][0]}')
print(edge_map)

15:17:07 - qiskit_qaoa.utils.sat_mapper - INFO - Num layers: 8
15:17:07 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:17:07 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:17:07 - qiskit_qaoa.utils.transpiler_passes - INFO - Loaded data
15:17:07 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:17:07 - qiskit_qaoa.utils.transpiler_passes - INFO - Loaded data
15:17:08 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 1596
15:17:08 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 67368
15:17:53 - qiskit_qaoa.utils.sat_mapper - INFO - [-79, -86, 97, 99, 100, 101, 102, 104, 106]
15:17:53 - qiskit_qaoa.utils.sat_mapper - INFO - [-86, -108, 112, 118]
15:17:54 - qiskit_qaoa.utils.sat_mapper - INFO - [-79, -124, 133, 134, 135, 137, 138, 140, 144]
15:17:54 - qiskit_qaoa.utils.sat_mapper - INFO - [-116, -124, 133, 134, 135, 137, 138, 139, 144]
15:17:54 - qiskit_qaoa.utils.sat_mapper - INFO - [-18, -35, 51, 5

In [26]:
edge_map = {0: 2, 1: 5, 2: 10, 3: 4, 4: 0, 5: 8, 6: 6, 7: 1, 8: 11, 9: 7, 10: 3, 11: 9}


In [ ]:
clauses = [
    [-79, -86, 97, 99, 100, 101, 102, 104, 106], # (6,7,8) -> (1,6,11)
    [-86, -108, 112, 118], # (7,8,9) -> (1,11,7) ?? 
    [-79, -124, 133, 134, 135, 137, 138, 140, 144], # (6,10,11) -> (6,3,9) ??
    [-116, -124, 133, 134, 135, 137, 138, 139, 144], # (9,10,11) -> (7,3,9) 
    [-18, -35, 51, 53, 55, 56, 57, 58, 60], # (1,2,4) -> (5,10,0) ??
    [-35, -41, 51, 54, 56, 57, 58], # (2,3,4) -> (10,4,0)
    [-3, -18, 61, 62, 64, 65, 67, 68, 71], # (0,1,5) -> (2,5,8) 
    [-3, -41, 61, 62, 66, 68, 71] # (0,3,5) -> (2,4,8) ??
]
for clause in clauses:
    print( (np.abs(clause) -1) // 12)
    print( ((np.abs(clause) - 1) % 12) * np.sign(clause))
    print()

[6 7 8 8 8 8 8 8 8]
[-6 -1  0  2  3  4  5  7  9]

[7 8 9 9]
[ -1 -11   3   9]

[ 6 10 11 11 11 11 11 11 11]
[-6 -3  0  1  2  4  5  7 11]

[ 9 10 11 11 11 11 11 11 11]
[-7 -3  0  1  2  4  5  6 11]

[1 2 4 4 4 4 4 4 4]
[ -5 -10   2   4   6   7   8   9  11]

[2 3 4 4 4 4 4]
[-10  -4   2   5   7   8   9]

[0 1 5 5 5 5 5 5 5]
[-2 -5  0  1  3  4  6  7 10]

[0 3 5 5 5 5 5]
[-2 -4  0  1  5  7 10]



In [27]:
new_hamiltonian = hamiltonian.apply_layout([edge_map[i] for i in range(num_qubits)], num_physical_qubits)

In [28]:
pm = PassManager(
    [
        # HighLevelSynthesis(basis_gates=["PauliEvolution"]), # Not needed if set up circuit as PauliEvolutionGate
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            extended_swap_strat,
            max_layers=num_layers,
            perform_extra_swaps=False
        ),
        SwapToFinalMapping(),
        DecomposePauliZEvolution(extended_swap_strat._coupling_map),
        # HighLevelSynthesis(
        #     basis_gates=["sx", "x", "rz", "rzz", "cx", "id", "swap"], 
        #     coupling_map=extended_swap_strat._coupling_map, 
        #     use_qubit_indices=True,
        #     qubits_initially_zero=False
        # ),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [29]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(new_hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

15:18:10 - qiskit_qaoa.utils.transpiler_passes - INFO - Max layers needed to apply swap decompose: 8
15:18:10 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 4
15:18:10 - qiskit_qaoa.utils.transpiler_passes - INFO - [(3, 7, 9), (2, 5, 8), (0, 4, 10), (1, 6, 11)]
15:18:10 - qiskit_qaoa.utils.transpiler_passes - INFO - Not implementing those gates


15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops
15:18:10 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops


In [45]:
extended_swap_strat.distance_nodes((1,7,11))

6

In [56]:
dt3 = extended_swap_strat.distance_tensor(3)

In [57]:
dt3[1,7,11]

np.int64(6)

In [ ]:
# 0 1 2 3
# 4 5 6 7
# 8 9 10 11

In [54]:
list(extended_swap_strat.all_connected_subgraphs(6, 3))

[(0, 9, 2),
 (0, 9, 3),
 (0, 9, 5),
 (0, 9, 8),
 (0, 9, 10),
 (0, 2, 8),
 (0, 2, 3),
 (0, 3, 1),
 (0, 3, 10),
 (1, 11, 10),
 (1, 11, 3),
 (1, 11, 7),
 (1, 3, 10),
 (2, 8, 9),
 (2, 8, 4),
 (3, 10, 9),
 (3, 10, 11),
 (3, 10, 6),
 (4, 8, 9),
 (4, 8, 5),
 (4, 5, 9),
 (4, 5, 6),
 (5, 9, 8),
 (5, 9, 10),
 (5, 9, 6),
 (5, 6, 10),
 (5, 6, 7),
 (6, 10, 9),
 (6, 10, 11),
 (6, 10, 7),
 (6, 7, 11),
 (7, 11, 10),
 (8, 9, 10),
 (9, 10, 11)]

In [55]:
extended_swap_strat._distances[(1,7,11)]

6

In [50]:
list(extended_swap_strat.swapped_coupling_map(6))

[(7, 11),
 (7, 6),
 (11, 7),
 (6, 7),
 (6, 10),
 (6, 5),
 (10, 6),
 (5, 6),
 (5, 9),
 (5, 4),
 (9, 5),
 (4, 5),
 (11, 1),
 (11, 10),
 (1, 11),
 (10, 11),
 (10, 3),
 (10, 9),
 (3, 10),
 (9, 10),
 (9, 0),
 (9, 8),
 (0, 9),
 (8, 9),
 (1, 3),
 (3, 1),
 (3, 0),
 (0, 3),
 (0, 2),
 (2, 0),
 (4, 8),
 (8, 4),
 (8, 2),
 (2, 8)]